# DWthon Adaptive Planning: Dzień 3

## Jak sprawdzić, że autonomia działa, ale nie wymyka się spod kontroli?

Dzisiaj prawie nie piszemy kodu. To celowe. Wczoraj zobaczyłeś silnik od środka. Dziś ten sam silnik dostaje interfejs, a Ty sprawdzasz jego zachowanie jak właściciel produktu i architekt systemu. Przejdziemy przez wspólny, zapisany scenariusz z dokładnie siedmioma turami. Dane wejściowe są przygotowane, ale w każdej turze prawdziwy API Call do modelu wybiera akcję z ograniczonego menu i podaje krótkie uzasadnienie. Zobaczysz zmianę planu, kontrargument, jedno pytanie do człowieka i ograniczony pilot.

## Gdzie jesteśmy: pętla jest już gotowa

W dniu 1 zobaczyłeś, że plan ma być hipotezą. W dniu 2 zobaczyłeś trzy ważne bramki: robot może zmienić plan po danych, cytat musi istnieć w źródle, a silna rekomendacja potrzebuje dowodów. Dzisiaj nie budujemy kolejnego silnika. Dajemy temu samemu pomysłowi ekran, na którym możesz sprawdzić go jak właściciel produktu.

To jest zasada, którą warto wynieść z warsztatu. Interfejs nie naprawi bałaganu w stanie. Ładnie narysuje ten sam bałagan. Najpierw silnik, potem produkt.

## Mapa finału

```text
Twoje pytanie
    |
    v
Kontrakt: cel, horyzont, dowód, budżet
    |
    v
Plan z pytaniami -> NextStep -> dispatcher -> obserwacja
    |                                      |
    +-------------- oś decyzji ------------+
    |
    v
Raport Robota: rekomendacja, dowody, droga, założenia, luki
```

W UI zobaczysz cały ten przepływ jednocześnie. To jest efekt wow. Nie dlatego, że mamy kolorowe karty, ale dlatego, że możesz kliknąć decyzję robota i wrócić do stanu, na podstawie którego ją podjął.

## Dwa pliki, dwie odpowiedzialności

```text
day3.ipynb             day3_runtime.py                 app.py
przewodnik         ->  silnik, API i zasady        ->  Gradio
co testujemy           jak robot działa                 jak to widzisz i klikasz
```

Do skopiowania wystarczą dwa pliki Pythona: `day3_runtime.py` i `app.py`. Możesz później zamienić Gradio na Slacka, FastAPI albo cron bez przepisywania pętli. Jeśli zmiana UI wymaga naruszenia silnika, granica między produktem a logiką była tylko udawana.

## Dzisiaj patrzysz jak właściciel produktu

Nie musisz analizować długiego logu ani przepisywać kodu. Najpierw uruchomisz scenariusz DEMO i sprawdzisz trzy rzeczy: czy robot dobrze rozumie cel, czy pokazuje podstawę kolejnej decyzji oraz czy umie uczciwie powiedzieć, czego nie wie.

To ważniejsze niż kolor kart. Interfejs ma sprawić, że możesz zakwestionować robota bez zaglądania do jego głowy.

In [ ]:
# Dzisiaj kod jest już gotowy. Pierwszy krok to uruchomienie aplikacji niżej.


## 🆘 Zanim odpalisz serwer

1. Uruchom komórkę z `app.py` i jej nie zamykaj.
2. Poczekaj, aż log potwierdzi uruchomienie aplikacji.
3. Dopiero wtedy otwórz link do Gradio w nowej karcie.
4. Jeśli aplikacja nie startuje, sprawdź komunikat w tej komórce i poproś o pomoc na <a href="https://dataworkshopclub.slack.com/archives/C0AD0KN2SMR">#dwthon_llm_day3_done</a>.
5. Jeśli port jest zajęty, zatrzymaj poprzednią komórkę i uruchom ją ponownie.

Aplikacja ma własny stan w RAM. Restart daje czysty przebieg demo. To nie jest wada. Dzięki temu każdy test zaczyna się od tych samych danych.

## Co uruchamiasz dzisiaj

Aplikacja nazywa się "Mój Robot". Dostaje jeden wspólny kontrakt i na końcu oddaje Raport Robota. Pomiędzy nimi nie ma czarnej skrzynki: widzisz kontrakt, plan z pytaniami, dowody, alerty i oś decyzji.

To ważne, bo końcowy raport nie jest jedynym wynikiem pracy. Równie wartościowa jest droga: co robot widział, kiedy zmienił plan, co świadomie odrzucił oraz czego nie sprawdził. To właśnie tę drogę będziesz dziś testować.

### Jeden scenariusz, bez przypadku

Dane w scenariuszu są przygotowane, więc każda osoba pracuje na tym samym materiale. Nie oznacza to mocka decyzji: w każdej turze model dostaje stan, wybiera akcję z małego menu i zwraca krótkie uzasadnienie. Jeśli zaproponuje akcję złą dla tego etapu, kod ją odrzuca, pokazuje powód i model wybiera ponownie. W UI zobaczysz: co model wybrał, z jakich opcji oraz dlaczego.

In [ ]:
# Uruchom aplikację w osobnym procesie. Nie zamykaj tej komórki.
!python app.py


## 👉 [Mój Robot na porcie 7860](/hub/user-redirect/proxy/7860/) 👈

Po uruchomieniu komórki wyżej Gradio wypisze lokalny adres. Otwórz go w nowej karcie. Jeśli pracujesz na JupyterHub, kliknij duży nagłówek powyżej. Nie tworzymy publicznego linku `gradio.live`.

Nie zamykaj komórki z `app.py`. To osobny proces, który trzyma aplikację. Gdy ją zatrzymasz, UI przestanie działać i możesz uruchomić je ponownie z czystym stanem.

## Jak czytać ekran Mojego Robota

Ekran ma pięć miejsc. Na górze są kontrakt i plan: co jest granicą zadania oraz jakie pytania robot uznaje za aktualne. Niżej po lewej rośnie oś decyzji. Po prawej są dowody z alertami polityki oraz Raport Robota. Na końcu raport zbiera rekomendację i otwarte pytanie.

Nie próbuj czytać wszystkiego naraz. Przy każdym teście zadaj sobie dwa pytania: "co się zmieniło w stanie?" i "czy robot miał prawo podjąć tę decyzję?". To wystarczy, żeby zobaczyć architekturę w działaniu.

```text
KONTRAKT       OŚ DECYZJI        PLAN v1 -> vN
user/default   T1 ... T7         + dodane, - porzucone
inferred       co robot widział  powód zmiany

DOWODY          ALERTY            RAPORT ROBOTA
opinia/repo     blokady           rekomendacja + luki
case/kontra     odrzucenia        droga + założenia
```

Przycisk "Co robot widział" pokazuje dokładny stan przekazany modelowi w danej turze. To nie jest ozdobnik. Jeśli ktoś pyta, skąd agent wziął decyzję, masz konkretną odpowiedź zamiast opowieści po fakcie.

## Test 1: czy robot zrozumiał decyzję?

Uruchom robota. Zanim przeczytasz oś decyzji, spójrz na kontrakt po lewej: cel, horyzont i standard dowodu. To jest chwila, w której sprawdzasz, czy robot pracuje nad decyzją, a nie nad ładniejszą wersją własnego problemu.

Jeśli kontrakt nie pasuje, nie ufaj dalszej części raportu. Dobry robot nie zaczyna od odpowiedzi. Najpierw pokazuje, co rozumie jako zadanie.

In [ ]:
# W Gradio kliknij: Uruchom scenariusz Mojego Robota.
# Najpierw przeczytaj kontrakt, dopiero potem oś decyzji.


## Test 2: przeczytaj jedną decyzję, nie cały ekran

Na osi decyzji wybierz jedną turę i przeczytaj tylko trzy rzeczy: co robot zaobserwował, co zmienił w planie oraz jaką akcję wybrał. To jest minimalny test zaufania. Czy kolejna akcja wynika z danych, które widzisz?

Nie musisz zgadzać się z robotem. Właśnie o to chodzi: masz wystarczająco dużo kontekstu, żeby go zakwestionować.

In [ ]:
# Wybierz jedną kartę na osi decyzji.
# Odpowiedz sobie: czy ta akcja ma sens po tej obserwacji?


## Test 3: zobacz granicę, nie tylko odpowiedź

Spójrz na panel "Dowody i granice". Jeśli robot ma same wyniki wyszukiwania lub opinie, nie powinien zachowywać się tak, jakby miał pewność. Alert nie jest błędem UI. To informacja: czego jeszcze brakuje, żeby rekomendacja była uczciwa.

Najważniejsze pytanie nie brzmi: "czy raport brzmi mądrze?". Brzmi: "czy raport pokazuje, czego jeszcze nie wiemy?".

In [ ]:
# W panelu Dowody i granice przeczytaj alerty polityki.
# Porównaj je z końcowym Raportem Robota.


## Test 4: gdy robot potrzebuje Ciebie

Robot zatrzyma się raz z pytaniem i pokaże dwie opcje odpowiedzi. To nie jest awaria. To znak, że kilka dróg ma sens, ale wybór zależy od kontekstu, którego nie ma w danych. Wybierz odpowiedź i kliknij "Kontynuuj pracę robota".

W tym scenariuszu pytanie pojawia się dokładnie raz: po zebraniu karty dowodowej i kontrargumentu. Dzięki temu możesz zobaczyć, że odpowiedź człowieka zmienia dalszą pracę, a nie tylko końcowe brzmienie raportu.

In [ ]:
# Jeśli pojawi się pytanie od robota, wybierz odpowiedź i kliknij Kontynuuj pracę robota.


## Test 5: granice nie są tylko tekstem w prompcie

W każdej turze na osi zobaczysz linię "Po tej obserwacji robot może". To nie jest ozdobnik. Pokazuje przestrzeń decyzji, którą system dał modelowi po otrzymaniu danych z tej tury. Gdy brakuje dowodów, dobra aplikacja nie udaje pewności. Gdy akcja nie ma już sensu, nie powinna być dostępna bez końca.

To jest różnica między prośbą "bądź ostrożny" a architekturą, która naprawdę ogranicza ryzyko.

In [ ]:
# Na osi znajdź linię: Po tej obserwacji robot może. Przeczytaj ją przy jednej turze.


## Druga ścieżka: zmień priorytet człowieka

Uruchom scenariusz drugi raz. Gdy robot zapyta o priorytet, wybierz inną odpowiedź. Nie zmienia się cel ani dane wejściowe. Zmienia się tylko to, co dla zespołu jest ważniejsze. Zobaczysz, że inny będzie zakres pilota i jego miara sukcesu.

To jest ważniejszy test niż własne, przypadkowe pytanie. Pokazuje, że człowiek nie jest ozdobnym przyciskiem: jego decyzja steruje kolejnym krokiem robota.

## Przeczytaj Raport Robota

Raport to nie tylko rekomendacja. Sprawdź też sekcję "Czego robot nie wie". To ona mówi, gdzie kończy się autonomia i co nadal wymaga wiedzy Twojego zespołu. Dobry raport można zakwestionować, bo pokazuje zarówno podstawę, jak i lukę.

In [ ]:
# W aplikacji przeczytaj Raport Robota oraz sekcję Czego robot nie wie.


## Co sprawdzać?

Po uruchomieniu robota odpowiedz sobie na trzy pytania: czy kontrakt odpowiadał Twojej decyzji? Czy przy jednej turze rozumiałeś, dlaczego robot zrobił następny krok? Czy raport uczciwie nazwał lukę? Jeśli tak, nie oglądałeś tylko UI. Sprawdziłeś, czy autonomia jest widoczna i możliwa do zakwestionowania.

## 🧠 Zadanie 3.1: znajdź moment, w którym robot zmienia zdanie

Kliknij "Uruchom scenariusz Mojego Robota". Nie czytaj całego ekranu naraz. Znajdź tylko turę, w której z planu znika pytanie o katalog narzędzi, a pojawia się pytanie o klasy zmian.

Potem odpowiedz na jedyne pytanie robota, wybierając `Mniej szumu dla reviewerów`, i kliknij "Kontynuuj pracę robota". Zobacz, jak powstaje pilot dla małych, dobrze testowanych PR-ów: bez automatycznego merge, z miarami czasu review, akceptacji sugestii i fałszywych pozytywów.

Twoim wynikiem jest jedno zdanie: "Robot porzucił pytanie o katalog narzędzi, ponieważ ...".

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈</summary>

Szukaj karty z komunikatem "Sygnał i szum są ważniejsze niż sama lista narzędzi". To podpowiedź, że robot nie zmienia celu, tylko drogę do celu. W punkcie pytania do człowieka wybierz dokładnie `Mniej szumu dla reviewerów` - ta ścieżka będzie punktem odniesienia dla Zadania 3.2.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈</summary>

W turze 3 robot porzuca pytanie "Jakie narzędzia są dostępne?", bo zebrany sygnał pokazał ważniejszy problem: sugestie pomagają przy małych, dobrze testowanych PR-ach, a przy dużych zmianach tworzą szum. W turze 5 pyta człowieka o miarę sukcesu tylko raz. Po wyborze `Mniej szumu dla reviewerów` w turze 6 powstaje pilot dla małych, dobrze testowanych PR-ów, mierzony czasem review, akceptacją sugestii i fałszywymi pozytywami. Tura 7 daje `ZBADAJ`, nie globalne wdrożenie. To jest adaptive planning: obserwacja zmienia pytanie, a decyzja człowieka zmienia działanie.

</details>
</details>

## 🧠 Zadanie 3.2: zmień jedną decyzję człowieka

Uruchom scenariusz jeszcze raz. W Zadaniu 3.1 punktem odniesienia był wybór `Mniej szumu dla reviewerów`. Tym razem, gdy robot zapyta o priorytet, wybierz `Wykrywanie większej liczby ryzyk`. Porównaj turę 6 i Raport Robota z poprzednim przebiegiem.

Twoim wynikiem jest jedno zdanie: "Po zmianie priorytetu z mniej szumu na więcej ryzyk robot zmienił ...". Nie patrz tylko na słowa w raporcie. Sprawdź trzy konkretne różnice: zakres pilota, zasadę bezpieczeństwa i miarę sukcesu.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈</summary>

Szukaj trzech miejsc: "zakres", "zasada bezpieczeństwa" i "miara sukcesu". To trzy części pilota, które powinny zmienić się po innej odpowiedzi człowieka.

<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈</summary>

Przy priorytecie `Mniej szumu dla reviewerów` robot ogranicza pilot do małych, dobrze testowanych PR-ów. Sugestie są komentarzami, bez automatycznego merge, a miary to czas review, odsetek zaakceptowanych sugestii i fałszywe pozytywy. Przy priorytecie `Wykrywanie większej liczby ryzyk` zawęża pilot do PR-ów dotyczących autoryzacji i bezpieczeństwa. Każdą sugestię zatwierdza doświadczony reviewer, a miary to liczba zweryfikowanych ryzyk bezpieczeństwa oraz fałszywe pozytywy. To nie jest inna odpowiedź napisana innymi słowami. To inny eksperyment.

</details>
</details>

## Co jest prawdziwe, a czego ten robot jeszcze nie wie

Prawdziwy jest wzorzec: kontrakt, pytania badawcze, typowana decyzja, dispatcher, walidacja granic, historia i raport. W każdej turze model wykonuje prawdziwy API Call: wybiera jedną akcję z ograniczonego menu i podaje krótkie uzasadnienie. Kod sprawdza tę decyzję przed wykonaniem - gdy łamie granicę scenariusza, pokazuje odrzucenie i prosi model o wybór zgodny z polityką.

Przygotowane są natomiast dane wejściowe i punkty zwrotne. Dzięki temu DEMO nie udaje aktualnego researchu z internetu, lecz jest małym, kontrolowanym laboratorium: można zobaczyć, jak obserwacja zmienia plan, a odpowiedź człowieka zmienia zakres pilota. Własne źródła, pamięć i harmonogram to kolejne moduły prawdziwego Mojego Robota.

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie Dnia 3</h2>

<h4>Co masz teraz zrobić? (3 kroki)</h4>

<ol>
  <li>🧪 <b>Uruchom DEMO Mojego Robota</b> i odpowiedz na jedno pytanie, które zada.</li>
  <li>📸 <b>Zrób screenshot</b> tury ze zmianą planu albo końcowego raportu z rekomendacją pilota.</li>
  <li>💬 <b>Wrzuć go na kanał <a href="https://dataworkshopclub.slack.com/archives/C0AD0KN2SMR">#dwthon_llm_day3_done</a></b> i dopisz: "Robot zmienił plan, bo ...".</li>
</ol>

Dopiero tutaj zamykamy całość. Masz nie tylko kod, ale coś, co możesz pokazać i przenieść do własnego procesu.

</div>

## 🎯 Dla chętnych: sprawdź, co naprawdę zmienia decyzja człowieka

Uruchom scenariusz drugi raz i przy pytaniu robota wybierz drugą odpowiedź. Nie porównuj samych słów w raporcie. Porównaj trzy rzeczy: zakres pilota, zasadę bezpieczeństwa i miarę sukcesu. To jest istota adaptive planning: ta sama wiedza wejściowa nie musi prowadzić do tego samego działania, kiedy zmienia się priorytet człowieka.

Jeśli udostępniasz Dzień 3 na LinkedIn, pokaż Raport Robota albo jeden fragment osi decyzji. Napisz o własnym odkryciu, np.: "Najbardziej wartościowe w Adaptive Planning nie jest to, że agent sam wybiera następny krok. Jest to, że po każdej decyzji widzę dlaczego i gdzie kończy się jego autonomia."

Dodaj baner wydarzenia z materiałów organizacyjnych, oznacz @DataWorkshop i @Vladimir Alekseichenko, a link lub screen wyślij na <a href="https://dataworkshopclub.slack.com/archives/C0AD0KN2SMR">#dwthon_llm_day3_done</a>.

## Podsumowanie całego DWthonu

1. **Cel i granice należą do człowieka.** To kontrakt, do którego robot wraca przez cały przebieg.
2. **Plan jest hipotezą z pytań.** Model może go poprawiać, gdy obserwacja zmienia sens dalszej pracy.
3. **`NextStep` daje jednej turze jasny kształt.** Obserwacja, decyzja, patch, akcja, nowa obserwacja.
4. **Schema-Guided Reasoning ogranicza przestrzeń decyzji.** Model ma autonomię, ale tylko w akcjach, które system dopuszcza teraz.
5. **Kod sprawdza fakty i granice.** Model interpretuje znaczenie oraz steruje drogą. Kod nie pisze za niego patcha.
6. **Autonomia jest wiarygodna tylko wtedy, gdy jest transparentna.** Oś decyzji, stan, alerty i raport pozwalają zobaczyć drogę.
7. **Zbudowałeś starter Mojego Robota.** Dzisiaj przechodzisz przez kontrolowany scenariusz. Dalej można dodać własne źródła, pamięć, harmonogram i guardrails dla własnego procesu.

Zbudowaliście pętlę. Agentic AI jest o tym, jak sprawić, żeby ta pętla nie robiła głupot, gdy nikt nie patrzy.